In [3]:
# Step 1: Import libraries
import re
import math
from collections import defaultdict, Counter

# Step 2: Create sample documents
documents = {
    1: "Machine learning is a part of artificial intelligence",
    2: "Artificial intelligence is used in robotics",
    3: "Machine learning helps in data science",
    4: "Data science uses Python programming",
    5: "Python is used for machine learning"
}

# Step 3: Preprocess documents
def preprocess(text):
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    return words

# Step 4: Create inverted index
inverted_index = defaultdict(set)
for doc_id, text in documents.items():
    words = preprocess(text)
    for word in words:
        inverted_index[word].add(doc_id)
print("INVERTED INDEX")
for word, doc_ids in sorted(inverted_index.items()):
    print(word, ":", sorted(doc_ids))

# Step 5: Boolean Search Functions
def AND(word1, word2):
    return inverted_index[word1] & inverted_index[word2]

def OR(word1, word2):
    return inverted_index[word1] | inverted_index[word2]

def NOT(word):
    all_docs = set(documents.keys())
    return all_docs - inverted_index[word]

# Step 6: Perform Boolean Search
print("\nBOOLEAN SEARCH")
print("Machine AND Learning:",
      sorted(AND("machine", "learning")))

print("Python OR Robotics:",
      sorted(OR("python", "robotics")))

print("NOT Python:",
      sorted(NOT("python")))

# Step 7: Calculate TF-IDF
N = len(documents)

# Calculate IDF
idf = {}
for word, doc_ids in inverted_index.items():
    idf[word] = math.log(N / len(doc_ids))

# Step 8: Calculate document TF-IDF scores
tfidf = {}
for doc_id, text in documents.items():
    words = preprocess(text)
    word_count = Counter(words)
    total_words = len(words)
    tfidf[doc_id] = {}
    for word, count in word_count.items():
        tf = count / total_words
        tfidf[doc_id][word] = tf * idf[word]

# Step 9: Ranked Retrieval
def search(query):
    query_words = preprocess(query)
    scores = {}
    for doc_id in documents:
        score = 0
        for word in query_words:
            if word in tfidf[doc_id]:
                score += tfidf[doc_id][word]
        scores[doc_id] = score

    ranked_results = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    print("\nRANKED SEARCH RESULTS")
    for doc_id, score in ranked_results:
        if score > 0:
            print(
                "Document:", doc_id,
                "| Score:", round(score, 4),
                "| Text:", documents[doc_id]
            )

# Step 10: Test TF-IDF Search
search("machine learning")

INVERTED INDEX
a : [1]
artificial : [1, 2]
data : [3, 4]
for : [5]
helps : [3]
in : [2, 3]
intelligence : [1, 2]
is : [1, 2, 5]
learning : [1, 3, 5]
machine : [1, 3, 5]
of : [1]
part : [1]
programming : [4]
python : [4, 5]
robotics : [2]
science : [3, 4]
used : [2, 5]
uses : [4]

BOOLEAN SEARCH
Machine AND Learning: [1, 3, 5]
Python OR Robotics: [2, 4, 5]
NOT Python: [1, 2, 3]

RANKED SEARCH RESULTS
Document: 3 | Score: 0.1703 | Text: Machine learning helps in data science
Document: 5 | Score: 0.1703 | Text: Python is used for machine learning
Document: 1 | Score: 0.1277 | Text: Machine learning is a part of artificial intelligence
